# Tests de mini-funcionalidades de validate_trips

Este notebook se usa para probar helpers y bloques internos de `validate_trips()` antes de hacer tests más integrados de la función completa.

Objetivo:
- verificar minifuncionalidades de forma aislada
- detectar errores de implementación temprano
- dejar una base fácil de portar después a pytest

Convenciones:
- los tests usan `assert`
- cuando una prueba necesita inspección visual, se acompaña con `display(...)`
- las pruebas de este notebook no reemplazan tests integrados 

## Bloque 0. Preparación

### Imports generales

In [1]:
import json
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd

### Imports del modulo

In [2]:
from pylondrina.schema import (
    DomainSpec,
    FieldSpec,
    TripSchema,
    TripSchemaEffective,
)

from pylondrina.reports import Issue, ValidationReport
from pylondrina.datasets import TripDataset
from pylondrina.errors import SchemaError, ValidationError

from pylondrina.validation import (
    ValidationOptions,
    validate_trips,
    check_required_columns,
    check_types_and_formats,
    check_constraints,
    check_domains,
    check_temporal_consistency,
    check_duplicates,
    apply_issue_truncation,
    build_validation_summary,
    _build_effective_nullable_by_field,
    _build_effective_domains_by_field,
    _build_temporal_context,
    _constraint_params_are_valid,
    _describe_received_params,
    _extract_domain_values,
    _build_issues_summary,
    _options_to_event_parameters,
)

### Helpers de apoyo para test

In [3]:
def show_ok(label: str):
    print(f"OK - {label}")


def assert_json_safe(obj, label: str = "object"):
    try:
        json.dumps(obj, default=str)
    except Exception as e:
        raise AssertionError(f"{label} no es JSON-safe: {e}") from e


def get_issue_codes(issues):
    return [i.code if hasattr(i, "code") else i.get("code") for i in issues]


def assert_issue_present(issues, code: str):
    codes = get_issue_codes(issues)
    assert code in codes, f"No se encontró el issue {code}. Codes actuales: {codes}"


def assert_issue_absent(issues, code: str):
    codes = get_issue_codes(issues)
    assert code not in codes, f"Se encontró inesperadamente el issue {code}. Codes actuales: {codes}"


def assert_counts_by_level(issues, *, errors=None, warnings=None, info=None):
    counts = {"error": 0, "warning": 0, "info": 0}
    for issue in issues:
        counts[issue.level] = counts.get(issue.level, 0) + 1

    if errors is not None:
        assert counts["error"] == errors, f"errors esperado={errors}, actual={counts['error']}"
    if warnings is not None:
        assert counts["warning"] == warnings, f"warnings esperado={warnings}, actual={counts['warning']}"
    if info is not None:
        assert counts["info"] == info, f"info esperado={info}, actual={counts['info']}"

### Configuración visual

In [4]:
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 120)

print("Imports OK")
show_ok("Sección 0 cargada")

Imports OK
OK - Sección 0 cargada


## Bloque 1. Utilidades puras y construcción de contexto

Qué se prueba en este bloque: helpers base que sostienen el contrato de OP-02, pero que todavía no dependen del pipeline completo. Aquí conviene probar primero:

nullable efectivo,
parámetros de constraints,
extracción y precedencia de dominios,
contexto temporal,
y serialización/resumen mínimo para evento.

### Fixtures reutilizables minimas

In [5]:
def make_field(
    name: str,
    dtype: str,
    *,
    required: bool = False,
    constraints: dict | None = None,
    domain: DomainSpec | None = None,
) -> FieldSpec:
    return FieldSpec(
        name=name,
        dtype=dtype,
        required=required,
        constraints=constraints,
        domain=domain,
    )


def make_trip_schema(fields: list[FieldSpec], *, version: str = "1.1") -> TripSchema:
    return TripSchema(
        version=version,
        fields={f.name: f for f in fields},
        required=[f.name for f in fields if f.required],
        semantic_rules=None,
    )

### Test 1.2 - _build_effective_nullable_by_field

Qué prueba:
Que la nulabilidad efectiva siga exactamente la regla cerrada para OP-02:

- si existe constraints["nullable"], manda ese valor,
- si no existe:
    - required=True -> False
    - required=False -> True

In [6]:
schema_nullable = make_trip_schema([
    make_field("movement_id", "string", required=True),
    make_field("mode", "categorical", required=False),
    make_field("purpose", "categorical", required=False, constraints={"nullable": False}),
    make_field("comment", "string", required=True, constraints={"nullable": False}),
])

nullable_map = _build_effective_nullable_by_field(schema_nullable)

assert nullable_map["movement_id"] is False
assert nullable_map["mode"] is True
assert nullable_map["purpose"] is False
assert nullable_map["comment"] is False

show_ok("Test 1.2 - _build_effective_nullable_by_field")

OK - Test 1.2 - _build_effective_nullable_by_field


### Test 1.3 - _constraint_params_are_valid y _describe_received_params

Qué prueba:
Que un constraint conocido:

- acepte payload bien formado,
- rechace payload mal formado,
- y describa correctamente qué recibió para después poder emitir warning + skip.
Esto es importante porque el contrato vigente cerró explícitamente que params inválidos de constraint conocido no son fatales: se emite warning y se omite solo ese constraint.

In [7]:
# Casos válidos
assert _constraint_params_are_valid("nullable", True) is True
assert _constraint_params_are_valid("pattern", r"^[A-Z]+$") is True
assert _constraint_params_are_valid("range", {"min": 0, "max": 10}) is True
assert _constraint_params_are_valid("length", {"min": 2, "max": 8}) is True
assert _constraint_params_are_valid("datetime", {"allow_naive": False, "min": "2026-01-01"}) is True
assert _constraint_params_are_valid("h3", {"require_valid": True, "resolution": 8}) is True
assert _constraint_params_are_valid("unique", True) is True

# Casos inválidos
assert _constraint_params_are_valid("nullable", {"value": True}) is False
assert _constraint_params_are_valid("range", {"minimum": 0}) is False
assert _constraint_params_are_valid("length", {"min": "a"}) is False
assert _constraint_params_are_valid("datetime", {"allow_naive": "no"}) is False
assert _constraint_params_are_valid("h3", {"resolution": "8"}) is False
assert _constraint_params_are_valid("unique", "yes") is False

# Descripción de params recibidos
assert _describe_received_params({"min": 0, "max": 10}) == ["min", "max"]
assert _describe_received_params(True) == "bool"
assert _describe_received_params("abc") == "str"

show_ok("Test 1.3 - _constraint_params_are_valid + _describe_received_params")

OK - Test 1.3 - _constraint_params_are_valid + _describe_received_params


### Test 1.4 - _extract_domain_values

Qué prueba:
Que la extracción de dominios soporte tanto la forma simple como la forma rica, porque OP-02 debe leer dominios desde schema_effective y/o metadata["domains_effective"] sin inventar un atributo separado en el dataset.

In [8]:
simple_domain = ["walk", "bus", "metro"]
rich_domain = {
    "values": ["walk", "bus", "metro", "unknown"],
    "extendable": True,
    "extended": False,
    "added_values": [],
}
missing_values_domain = {"extendable": True}

assert _extract_domain_values(simple_domain) == {"walk", "bus", "metro"}
assert _extract_domain_values(rich_domain) == {"walk", "bus", "metro", "unknown"}
assert _extract_domain_values(missing_values_domain) is None
assert _extract_domain_values(None) is None

show_ok("Test 1.4 - _extract_domain_values")

OK - Test 1.4 - _extract_domain_values


### Test 1.5 - _build_effective_domains_by_field

Qué prueba:
La precedencia cerrada para dominios:

1. schema_effective
2. metadata["domains_effective"]
3. schema base

In [9]:
schema_domains = make_trip_schema([
    make_field(
        "mode",
        "categorical",
        required=False,
        domain=DomainSpec(values=["walk", "bus", "metro"]),
    ),
    make_field(
        "purpose",
        "categorical",
        required=False,
        domain=DomainSpec(values=["work", "study"]),
    ),
    make_field("user_id", "string", required=True),
])

schema_effective = TripSchemaEffective(
    domains_effective={
        "mode": {
            "values": ["walk", "bus", "metro", "bike"],
            "extendable": True,
            "extended": True,
            "added_values": ["bike"],
        }
    }
)

metadata = {
    "domains_effective": {
        "mode": {"values": ["walk", "bus"]},              # debe perder contra schema_effective
        "purpose": {"values": ["work", "study", "health"]},  # debe ganar sobre schema base
    }
}

effective_domains = _build_effective_domains_by_field(
    schema=schema_domains,
    schema_effective=schema_effective,
    metadata=metadata,
)

assert effective_domains["mode"] == {"walk", "bus", "metro", "bike"}
assert effective_domains["purpose"] == {"work", "study", "health"}

show_ok("Test 1.5 - _build_effective_domains_by_field")

OK - Test 1.5 - _build_effective_domains_by_field


### Test 1.6 - _build_temporal_context

Qué prueba:
Que el contexto temporal siga la precedencia correcta:

1. schema_effective.temporal
2. metadata["temporal"]
3. inferencia por columnas presentes
y que solo informe origin_time_utc/destination_time_utc en fields_present

In [10]:
df_t1 = pd.DataFrame({
    "origin_time_utc": ["2026-01-01T08:00:00Z"],
    "destination_time_utc": ["2026-01-01T08:30:00Z"],
})

df_t2 = pd.DataFrame({
    "origin_time_local_hhmm": ["08:00"],
    "destination_time_local_hhmm": ["08:30"],
})

# 1) schema_effective manda
ctx = _build_temporal_context(
    df=df_t1,
    metadata={"temporal": {"tier": "tier_2"}},
    schema_effective=SimpleNamespace(temporal={"tier": "tier_1"}),
)
assert ctx["tier"] == "tier_1"
assert ctx["fields_present"] == ["origin_time_utc", "destination_time_utc"]

# 2) metadata manda si schema_effective no trae temporal
ctx = _build_temporal_context(
    df=df_t1,
    metadata={"temporal": {"tier": "tier_2"}},
    schema_effective=SimpleNamespace(),
)
assert ctx["tier"] == "tier_2"

# 3) inferencia por columnas si no hay contexto previo
ctx = _build_temporal_context(
    df=df_t1,
    metadata={},
    schema_effective=SimpleNamespace(),
)
assert ctx["tier"] == "tier_1"

ctx = _build_temporal_context(
    df=df_t2,
    metadata={},
    schema_effective=SimpleNamespace(),
)
assert ctx["tier"] == "tier_2"

ctx = _build_temporal_context(
    df=pd.DataFrame({"user_id": ["u1"]}),
    metadata={},
    schema_effective=SimpleNamespace(),
)
assert ctx["tier"] == "tier_3"

show_ok("Test 1.6 - _build_temporal_context")

OK - Test 1.6 - _build_temporal_context


### Test 1.7 - _options_to_event_parameters

Qué prueba:
Que la serialización de parámetros para evento:

- sea JSON-safe,
- y convierta duplicates_subset de tuple a list.

In [11]:
options = ValidationOptions(
    strict=True,
    validate_duplicates=True,
    duplicates_subset=("user_id", "origin_time_utc"),
    allow_partial_od_spatial=True,
)

params = _options_to_event_parameters(options)

assert params["strict"] is True
assert params["validate_duplicates"] is True
assert params["duplicates_subset"] == ["user_id", "origin_time_utc"]
assert params["allow_partial_od_spatial"] is True
assert_json_safe(params, "event_parameters")

show_ok("Test 1.7 - _options_to_event_parameters")

OK - Test 1.7 - _options_to_event_parameters


### Test 1.8 - _build_issues_summary

Qué prueba:
Que el resumen compacto para evento:

- cuente bien por severidad,
- agregue por code,
- y ordene top_codes por frecuencia descendente.
Esto es parte explícita del contrato observable de OP-02 en metadata/evento.

In [13]:
issues_for_summary = [
    Issue(level="warning", code="VAL.DOMAIN.PARTIAL_COVERAGE", message="warn 1"),
    Issue(level="warning", code="VAL.DOMAIN.PARTIAL_COVERAGE", message="warn 2"),
    Issue(level="error", code="VAL.CORE.NULLABILITY_VIOLATION", message="err 1"),
    Issue(level="info", code="VAL.DEBUG.SOMETHING", message="info 1"),
]

issues_summary = _build_issues_summary(issues_for_summary)

assert issues_summary["counts"] == {"info": 1, "warning": 2, "error": 1}
assert issues_summary["top_codes"][0] == {
    "code": "VAL.DOMAIN.PARTIAL_COVERAGE",
    "count": 2,
}
assert_json_safe(issues_summary, "issues_summary")

show_ok("Test 1.8 - _build_issues_summary")

OK - Test 1.8 - _build_issues_summary


In [14]:
show_ok("Bloque 1 completo")

OK - Bloque 1 completo


## Bloque 2: tests de helpers principales

### Imports mínimos para esta sección

In [15]:
from pylondrina.schema import DomainSpec, TripSchema
from pylondrina.reports import Issue
from pylondrina.validation import (
    check_required_columns,
    check_types_and_formats,
    check_constraints,
    check_domains,
    check_temporal_consistency,
    check_duplicates,
    apply_issue_truncation,
    build_validation_summary,
    _build_effective_nullable_by_field,
)

### Grupo 2.1 - check_required_columns

Qué se prueba:
Que el helper detecte columnas requeridas ausentes y emita el code agregado correcto, y que no emita nada cuando el dataframe cumple. Este bloque representa el primer check formal del contrato de OP-02.

#### 2.1.1 - Caso feliz sin faltantes

In [16]:
schema_required = make_trip_schema([
    make_field("movement_id", "string", required=True),
    make_field("user_id", "string", required=True),
    make_field("mode", "categorical", required=False),
])

df_required_ok = pd.DataFrame({
    "movement_id": ["m1", "m2"],
    "user_id": ["u1", "u2"],
    "mode": ["walk", "bus"],
})

issues = check_required_columns(df_required_ok, schema=schema_required)

assert issues == []
show_ok("Grupo 2.1.1 - check_required_columns happy path")

OK - Grupo 2.1.1 - check_required_columns happy path


#### 2.1.2 - faltan columnas requeridas

In [18]:
df_required_missing = pd.DataFrame({
    "movement_id": ["m1", "m2"],
    "mode": ["walk", "bus"],
})

issues = check_required_columns(df_required_missing, schema=schema_required)

assert len(issues) == 1
assert_issue_present(issues, "VAL.CORE.REQUIRED_COLUMNS_MISSING")
assert_counts_by_level(issues, errors=1, warnings=0, info=0)

issue = issues[0]
assert issue.row_count == 1
assert issue.details["missing_required"] == ["user_id"]
assert issue.details["required"] == ["movement_id", "user_id"]

display(issue)
show_ok("Grupo 2.1.2 - check_required_columns con faltantes")

Issue(level='error', code='VAL.CORE.REQUIRED_COLUMNS_MISSING', message="Faltan columnas requeridas según el TripSchema: ['user_id'].", field=None, source_field=None, row_count=1, details={'check': 'required_columns', 'missing_required': ['user_id'], 'required': ['movement_id', 'user_id'], 'available_columns_sample': ['movement_id', 'mode'], 'available_columns_total': 2, 'action': 'report_error'})

OK - Grupo 2.1.2 - check_required_columns con faltantes


### Grupo 2.2 - check_types_and_formats

Qué se prueba:
Que el helper detecte valores no interpretables por dtype sin mutar el dataframe, y que agregue un issue por campo afectado. Este bloque cubre el check formal de tipos/formatos básicos de OP-02.

#### 2.2.1 - mezcla de dtypes con valores inválidos

In [20]:
schema_types = make_trip_schema([
    make_field("int_field", "int"),
    make_field("float_field", "float"),
    make_field("dt_field", "datetime"),
    make_field("bool_field", "bool"),
    make_field("cat_field", "categorical"),
    make_field("str_field", "string"),
])

df_types = pd.DataFrame({
    "int_field": [1, "2", "3.5", "abc", None],
    "float_field": [1.2, "2.3", "abc", None, 7],
    "dt_field": ["2026-01-01T08:00:00Z", "not-a-date", None, "2026-01-02", "13/99/2026"],
    "bool_field": [True, "false", "YES", "maybe", None],
    "cat_field": ["walk", "bus", None, "xxx", "metro"],  # categorical no falla aquí
    "str_field": ["a", 2, None, "b", object()],          # string no falla aquí
})

issues = check_types_and_formats(
    df_types,
    schema=schema_types,
    sample_rows_per_issue=3,
)

codes = [i.code for i in issues]
fields = sorted(i.field for i in issues)

assert len(issues) == 4
assert codes.count("VAL.CORE.TYPE_OR_FORMAT_INVALID") == 4
assert fields == ["bool_field", "dt_field", "float_field", "int_field"]
assert_counts_by_level(issues, errors=4, warnings=0, info=0)

display(issues)
show_ok("Grupo 2.2.1 - check_types_and_formats con múltiples campos inválidos")

[Issue(level='error', code='VAL.CORE.TYPE_OR_FORMAT_INVALID', message="El campo 'int_field' contiene valores no interpretables para el dtype/formato esperado ('int').", field='int_field', source_field=None, row_count=2, details={'check': 'types_and_formats', 'n_rows_total': 5, 'n_violations': 2, 'row_indices_sample': [2, 3], 'values_sample': ['3.5', 'abc'], 'raw_values_sample': ['3.5', 'abc'], 'field': 'int_field', 'dtype_expected': 'int', 'parse_fail_count': 2, 'total_count': 4, 'fail_rate': 0.5, 'action': 'report_error'}),
 Issue(level='error', code='VAL.CORE.TYPE_OR_FORMAT_INVALID', message="El campo 'float_field' contiene valores no interpretables para el dtype/formato esperado ('float').", field='float_field', source_field=None, row_count=1, details={'check': 'types_and_formats', 'n_rows_total': 5, 'n_violations': 1, 'row_indices_sample': [2], 'values_sample': ['abc'], 'raw_values_sample': ['abc'], 'field': 'float_field', 'dtype_expected': 'float', 'parse_fail_count': 1, 'total_co

OK - Grupo 2.2.1 - check_types_and_formats con múltiples campos inválidos


#### 2.2.2 - no muta el dataframe

In [21]:
df_before = df_types.copy(deep=True)

_ = check_types_and_formats(
    df_types,
    schema=schema_types,
    sample_rows_per_issue=3,
)

pd.testing.assert_frame_equal(df_types, df_before)
show_ok("Grupo 2.2.2 - check_types_and_formats no muta el dataframe")

OK - Grupo 2.2.2 - check_types_and_formats no muta el dataframe


### Grupo 2.3 - check_constraints

Qué se prueba:
Que este helper concentre correctamente:

- nullabilidad efectiva,
- regla de OD parcial,
- y constraints declarativas simples como range, pattern, length, unique, datetime, h3.
Esto es especialmente importante porque en OP-02 quedó cerrado que aquí vive la parte más delicada del contrato de validación.

In [26]:
schema_constraints = make_trip_schema([
    make_field("movement_id", "string", required=True),
    make_field("score", "float", required=False, constraints={"nullable": False, "range": {"min": 0, "max": 10}}),
    make_field("comment", "string", required=False, constraints={"length": {"min": 2, "max": 5}}),
])

df_constraints = pd.DataFrame({
    "movement_id": ["m1", "m2", "m3", "m4"],
    "score": [5, None, -1, 12],
    "comment": ["ok", "x", "123456", None],
})

nullable_map = _build_effective_nullable_by_field(schema_constraints)

issues = check_constraints(
    df_constraints,
    schema=schema_constraints,
    effective_nullable_by_field=nullable_map,
    allow_partial_od_spatial=False,
    skipped_constraints={},
    sample_rows_per_issue=5,
)

codes = [i.code for i in issues]
assert "VAL.CORE.NULLABILITY_VIOLATION" in codes
assert "VAL.CORE.CONSTRAINT_VIOLATION" in codes
assert_counts_by_level(issues, errors=3)

# 1 issue por nullabilidad de score
# 1 issue por range de score
# 1 issue por length de comment
assert len(issues) == 3

display(issues)
show_ok("Grupo 2.3.1 - check_constraints nullability + range + length")

[Issue(level='error', code='VAL.CORE.NULLABILITY_VIOLATION', message="El campo 'score' no admite nulos según nullable efectivo, pero se detectaron filas con valores faltantes.", field='score', source_field=None, row_count=1, details={'check': 'constraints', 'n_rows_total': 4, 'n_violations': 1, 'row_indices_sample': [1], 'values_sample': [None], 'raw_values_sample': [None], 'field': 'score', 'nullable_effective': False, 'action': 'report_error'}),
 Issue(level='error', code='VAL.CORE.CONSTRAINT_VIOLATION', message="El campo 'score' viola la constraint declarativa 'range'.", field='score', source_field=None, row_count=2, details={'check': 'constraints', 'n_rows_total': 4, 'n_violations': 2, 'row_indices_sample': [2, 3], 'values_sample': [-1.0, 12.0], 'raw_values_sample': [-1.0, 12.0], 'field': 'score', 'constraint': 'range', 'expected': {'min': 0, 'max': 10}, 'observed_sample': [-1.0, 12.0], 'action': 'report_error'}),
 Issue(level='error', code='VAL.CORE.CONSTRAINT_VIOLATION', message=

OK - Grupo 2.3.1 - check_constraints nullability + range + length


#### 2.3.2 - regla especial de OD parcial

In [27]:
schema_od = make_trip_schema([
    make_field("origin_latitude", "float", required=True),
    make_field("origin_longitude", "float", required=True),
    make_field("destination_latitude", "float", required=True),
    make_field("destination_longitude", "float", required=True),
])

df_od = pd.DataFrame({
    # fila 0: origen completo, destino ausente -> válida
    "origin_latitude": [-33.45, None, None],
    "origin_longitude": [-70.66, None, None],
    # fila 1: destino completo, origen ausente -> válida
    "destination_latitude": [None, -33.44, None],
    "destination_longitude": [None, -70.62, None],
})

nullable_map = _build_effective_nullable_by_field(schema_od)

issues = check_constraints(
    df_od,
    schema=schema_od,
    effective_nullable_by_field=nullable_map,
    allow_partial_od_spatial=True,
    skipped_constraints={},
    sample_rows_per_issue=5,
)

assert len(issues) == 1
assert_issue_present(issues, "VAL.CORE.OD_SPATIAL_BOTH_MISSING")
assert_counts_by_level(issues, errors=1)

display(issues)
show_ok("Grupo 2.3.2 - check_constraints con OD parcial")

[Issue(level='error', code='VAL.CORE.OD_SPATIAL_BOTH_MISSING', message='La regla de OD parcial falló: la fila no tiene ni origen espacial completo ni destino espacial completo en coordenadas.', field=None, source_field=None, row_count=1, details={'check': 'constraints', 'n_rows_total': 3, 'n_violations': 1, 'row_indices_sample': [2], 'values_sample': [{'origin_latitude': None, 'origin_longitude': None, 'destination_latitude': None, 'destination_longitude': None}], 'raw_values_sample': [{'origin_latitude': None, 'origin_longitude': None, 'destination_latitude': None, 'destination_longitude': None}], 'fields_checked': ['origin_latitude', 'origin_longitude', 'destination_latitude', 'destination_longitude'], 'allow_partial_od_spatial': True, 'action': 'report_error'})]

OK - Grupo 2.3.2 - check_constraints con OD parcial


#### 2.3.3 - constraint conocida pero marcada como skipped

In [28]:
schema_skipped = make_trip_schema([
    make_field("score", "float", constraints={"range": {"min": 0, "max": 10}}),
])

df_skipped = pd.DataFrame({
    "score": [-5, 99, 4],
})

issues = check_constraints(
    df_skipped,
    schema=schema_skipped,
    effective_nullable_by_field=_build_effective_nullable_by_field(schema_skipped),
    allow_partial_od_spatial=False,
    skipped_constraints={"score": {"range"}},
    sample_rows_per_issue=5,
)

assert issues == []
show_ok("Grupo 2.3.3 - check_constraints respeta skipped_constraints")

OK - Grupo 2.3.3 - check_constraints respeta skipped_constraints


### Grupo 2.4 - check_domains

Qué se prueba:
Que el helper:

- respete off/full/sample,
- use ratio sobre no nulos,
- emita warning o error según umbral,
- y construya correctamente el bloque domains del summary.
Eso quedó cerrado explícitamente en el contrato aterrizado de OP-02.

#### 2.4.1 - modo off

In [29]:
schema_domains = make_trip_schema([
    make_field("mode", "categorical", domain=DomainSpec(values=["walk", "bus", "metro"])),
])

df_domains = pd.DataFrame({
    "mode": ["walk", "bus", "bike", None],
})

issues, block = check_domains(
    df_domains,
    schema=schema_domains,
    effective_domains_by_field={"mode": {"walk", "bus", "metro"}},
    mode="off",
    sample_frac=0.5,
    min_in_domain_ratio=1.0,
    sample_rows_per_issue=5,
)

assert issues == []
assert block is None

show_ok("Grupo 2.4.1 - check_domains modo off")

OK - Grupo 2.4.1 - check_domains modo off


#### 2.4.2 - coverage parcial -> warning

In [31]:
issues, block = check_domains(
    df_domains,
    schema=schema_domains,
    effective_domains_by_field={"mode": {"walk", "bus", "metro"}},
    mode="full",
    sample_frac=0.5,
    min_in_domain_ratio=0.5,
    sample_rows_per_issue=5,
)

assert len(issues) == 1
assert_issue_present(issues, "VAL.DOMAIN.PARTIAL_COVERAGE")
assert_counts_by_level(issues, errors=0, warnings=1, info=0)

assert block["mode"] == "full"
assert block["min_required_ratio"] == 0.5
assert "mode" in block["fields"]
assert block["fields"]["mode"]["n_checked_non_null"] == 3
assert block["fields"]["mode"]["n_in_domain"] == 2

display(issues)
show_ok("Grupo 2.4.2 - check_domains coverage parcial")

[Issue(level='warning', code='VAL.DOMAIN.PARTIAL_COVERAGE', message="El campo 'mode' tiene cobertura parcial en dominio (0.6667); supera el mínimo, pero no alcanza cobertura completa.", field='mode', source_field=None, row_count=1, details={'check': 'domains', 'n_rows_total': 4, 'n_violations': 1, 'row_indices_sample': [2], 'values_sample': ['bike'], 'raw_values_sample': ['bike'], 'field': 'mode', 'mode': 'full', 'ratio_in_domain': 0.6666666666666666, 'min_required_ratio': 0.5, 'n_checked_non_null': 3, 'n_in_domain': 2, 'domain_values_sample': ['bus', 'metro', 'walk'], 'action': 'report_warning'})]

OK - Grupo 2.4.2 - check_domains coverage parcial


#### Celda 2.4.3 - coverage bajo el mínimo -> error

In [32]:
issues, block = check_domains(
    df_domains,
    schema=schema_domains,
    effective_domains_by_field={"mode": {"walk", "bus", "metro"}},
    mode="full",
    sample_frac=0.5,
    min_in_domain_ratio=0.9,
    sample_rows_per_issue=5,
)

assert len(issues) == 1
assert_issue_present(issues, "VAL.DOMAIN.RATIO_BELOW_MIN")
assert_counts_by_level(issues, errors=1, warnings=0, info=0)

display(issues)
show_ok("Grupo 2.4.3 - check_domains ratio bajo mínimo")

[Issue(level='error', code='VAL.DOMAIN.RATIO_BELOW_MIN', message="El campo 'mode' no cumple el mínimo requerido de cobertura en dominio (0.6667 < 0.9000).", field='mode', source_field=None, row_count=1, details={'check': 'domains', 'n_rows_total': 4, 'n_violations': 1, 'row_indices_sample': [2], 'values_sample': ['bike'], 'raw_values_sample': ['bike'], 'field': 'mode', 'mode': 'full', 'ratio_in_domain': 0.6666666666666666, 'min_required_ratio': 0.9, 'n_checked_non_null': 3, 'n_in_domain': 2, 'domain_values_sample': ['bus', 'metro', 'walk'], 'action': 'report_error'})]

OK - Grupo 2.4.3 - check_domains ratio bajo mínimo


#### 2.4.4 - campo categórico sin dominio usable

In [33]:
schema_missing_domain = make_trip_schema([
    make_field("purpose", "categorical"),
])

df_missing_domain = pd.DataFrame({
    "purpose": ["work", "study", "health"],
})

issues, block = check_domains(
    df_missing_domain,
    schema=schema_missing_domain,
    effective_domains_by_field={"purpose": None},
    mode="full",
    sample_frac=0.5,
    min_in_domain_ratio=1.0,
    sample_rows_per_issue=5,
)

assert len(issues) == 1
assert_issue_present(issues, "VAL.DOMAIN.MISSING_DOMAIN_INFO")
assert_counts_by_level(issues, errors=0, warnings=1, info=0)

assert block["fields"] == {}

display(issues)
show_ok("Grupo 2.4.4 - check_domains sin domain info")

[Issue(level='warning', code='VAL.DOMAIN.MISSING_DOMAIN_INFO', message="No existe información de dominio usable para el campo categórico 'purpose'; se omitirá su validación de dominio.", field='purpose', source_field=None, row_count=None, details={'check': 'domains', 'field': 'purpose', 'domain_source_attempted': ['schema_effective', 'metadata', 'schema'], 'reason': 'missing_domain_values', 'action': 'skip_field'})]

OK - Grupo 2.4.4 - check_domains sin domain info


### Grupo 2.5 - check_temporal_consistency

Qué se prueba:
La regla temporal v1.1 quedó cerrada solo para Tier 1 y solo sobre origin_time_utc <= destination_time_utc. Este helper también debe devolver bloque de summary temporal incluso cuando no evalúa.

#### 2.5.1 - Tier no evaluable

In [34]:
df_temporal_t2 = pd.DataFrame({
    "origin_time_local_hhmm": ["08:00", "09:00"],
    "destination_time_local_hhmm": ["08:30", "09:20"],
})

issues, block = check_temporal_consistency(
    df_temporal_t2,
    temporal_context={"tier": "tier_2"},
    sample_rows_per_issue=5,
)

assert issues == []
assert block["evaluated"] is False
assert block["reason"] == "temporal_tier_not_1"

show_ok("Grupo 2.5.1 - check_temporal_consistency no evaluable por tier")

OK - Grupo 2.5.1 - check_temporal_consistency no evaluable por tier


#### 2.5.2 - Tier 1 sin invalidaciones

In [35]:
df_temporal_ok = pd.DataFrame({
    "origin_time_utc": ["2026-01-01T08:00:00Z", "2026-01-01T09:00:00Z"],
    "destination_time_utc": ["2026-01-01T08:30:00Z", "2026-01-01T09:20:00Z"],
})

issues, block = check_temporal_consistency(
    df_temporal_ok,
    temporal_context={"tier": "tier_1"},
    sample_rows_per_issue=5,
)

assert issues == []
assert block["evaluated"] is True
assert block["n_checked"] == 2
assert block["n_violations"] == 0

show_ok("Grupo 2.5.2 - check_temporal_consistency happy path")

OK - Grupo 2.5.2 - check_temporal_consistency happy path


#### 2.5.3 - Tier 1 con invalidaciones

In [40]:
df_temporal_bad = pd.DataFrame({
    "origin_time_utc": ["2026-01-01T09:00:00Z", "2026-01-01T10:00:00Z"],
    "destination_time_utc": ["2026-01-01T08:30:00Z", "2026-01-01T09:50:00Z"],
})

issues, block = check_temporal_consistency(
    df_temporal_bad,
    temporal_context={"tier": "tier_1"},
    sample_rows_per_issue=5,
)

assert len(issues) == 1
assert_issue_present(issues, "VAL.TEMPORAL.ORIGIN_AFTER_DESTINATION")
assert_counts_by_level(issues, errors=1, warnings=0, info=0)

issue = issues[0]
assert issue.row_count == 2
assert issue.details["n_violations"] == 2
assert issue.details["row_indices_sample"] == [0, 1]

assert block["evaluated"] is True
assert block["n_checked"] == 2
assert block["n_violations"] == 2

show_ok("Grupo 2.5.3 - check_temporal_consistency con violaciones")

OK - Grupo 2.5.3 - check_temporal_consistency con violaciones


### Grupo 2.6 - check_duplicates

Qué se prueba:
Que el helper detecte duplicados sobre subset explícito y construya su bloque de summary. En OP-02 la política del subset default ya quedó fuera; aquí solo se prueba el helper puro con subset ya validado.

#### 2.6.1 - sin duplicados

In [41]:
df_dup_ok = pd.DataFrame({
    "user_id": ["u1", "u2", "u3"],
    "origin_time_utc": ["2026-01-01T08:00:00Z", "2026-01-01T09:00:00Z", "2026-01-01T10:00:00Z"],
    "origin_longitude": [-70.66, -70.67, -70.68],
})

issues, block = check_duplicates(
    df_dup_ok,
    duplicates_subset=("user_id", "origin_time_utc"),
    sample_rows_per_issue=5,
)

assert issues == []
assert block["evaluated"] is True
assert block["n_duplicate_rows"] == 0

show_ok("Grupo 2.6.1 - check_duplicates happy path")

OK - Grupo 2.6.1 - check_duplicates happy path


#### 2.6.2 - con duplicados

In [42]:
df_dup_bad = pd.DataFrame({
    "user_id": ["u1", "u1", "u2", "u3"],
    "origin_time_utc": ["2026-01-01T08:00:00Z", "2026-01-01T08:00:00Z", "2026-01-01T09:00:00Z", "2026-01-01T10:00:00Z"],
    "origin_longitude": [-70.66, -70.66, -70.67, -70.68],
})

issues, block = check_duplicates(
    df_dup_bad,
    duplicates_subset=("user_id", "origin_time_utc"),
    sample_rows_per_issue=5,
)

assert len(issues) == 1
assert_issue_present(issues, "VAL.DUPLICATES.ROWS_FOUND")
assert_counts_by_level(issues, errors=1, warnings=0, info=0)

assert block["evaluated"] is True
assert block["n_duplicate_rows"] == 2
assert block["duplicates_subset"] == ["user_id", "origin_time_utc"]

display(issues)
show_ok("Grupo 2.6.2 - check_duplicates con filas duplicadas")

[Issue(level='error', code='VAL.DUPLICATES.ROWS_FOUND', message="Se detectaron filas duplicadas según duplicates_subset=['user_id', 'origin_time_utc'].", field=None, source_field=None, row_count=2, details={'check': 'duplicates', 'n_rows_total': 4, 'n_violations': 2, 'row_indices_sample': [0, 1], 'values_sample': [{'user_id': 'u1', 'origin_time_utc': '2026-01-01T08:00:00Z'}, {'user_id': 'u1', 'origin_time_utc': '2026-01-01T08:00:00Z'}], 'raw_values_sample': [{'user_id': 'u1', 'origin_time_utc': '2026-01-01T08:00:00Z'}, {'user_id': 'u1', 'origin_time_utc': '2026-01-01T08:00:00Z'}], 'duplicates_subset': ['user_id', 'origin_time_utc'], 'duplicate_keys_sample': [{'user_id': 'u1', 'origin_time_utc': '2026-01-01T08:00:00Z'}, {'user_id': 'u1', 'origin_time_utc': '2026-01-01T08:00:00Z'}], 'action': 'report_error'})]

OK - Grupo 2.6.2 - check_duplicates con filas duplicadas


### Grupo 2.7 - apply_issue_truncation

Qué se prueba:
La política de truncamiento es parte observable del contrato de OP-02: issue final explícito y bloque limits consistente.

#### 2.7.1 - no hay truncamiento

In [45]:
issues_input = [
    Issue(level="warning", code="VAL.DOMAIN.PARTIAL_COVERAGE", message="warn 1"),
    Issue(level="error", code="VAL.CORE.NULLABILITY_VIOLATION", message="err 1"),
]

issues_out, limits = apply_issue_truncation(issues_input, max_issues=5)

assert len(issues_out) == 2
assert limits["issues_truncated"] is False
assert limits["n_issues_emitted"] == 2
assert limits["n_issues_detected_total"] == 2

display(issues_out)
show_ok("Grupo 2.7.1 - apply_issue_truncation sin truncamiento")

[Issue(level='warning', code='VAL.DOMAIN.PARTIAL_COVERAGE', message='warn 1', field=None, source_field=None, row_count=None, details=None),
 Issue(level='error', code='VAL.CORE.NULLABILITY_VIOLATION', message='err 1', field=None, source_field=None, row_count=None, details=None)]

OK - Grupo 2.7.1 - apply_issue_truncation sin truncamiento


#### 2.7.2 - con truncamiento

In [47]:
issues_input = [
    Issue(level="error", code=f"VAL.TEST.CODE_{i}", message=f"issue {i}")
    for i in range(5)
]

issues_out, limits = apply_issue_truncation(issues_input, max_issues=3)

codes = [i.code for i in issues_out]

assert len(issues_out) == 3
assert "VAL.CORE.ISSUES_TRUNCATED" in codes
assert limits["issues_truncated"] is True
assert limits["n_issues_emitted"] == 3
assert limits["n_issues_detected_total"] == 5

display(issues_out)
show_ok("Grupo 2.7.2 - apply_issue_truncation con truncamiento")

[Issue(level='error', code='VAL.TEST.CODE_0', message='issue 0', field=None, source_field=None, row_count=None, details=None),
 Issue(level='error', code='VAL.TEST.CODE_1', message='issue 1', field=None, source_field=None, row_count=None, details=None),
 Issue(level='warning', code='VAL.CORE.ISSUES_TRUNCATED', message='El reporte fue truncado por max_issues=3; no se emitieron todos los hallazgos detectados.', field=None, source_field=None, row_count=None, details={'check': 'limits', 'max_issues': 3, 'n_issues_emitted': 3, 'n_issues_detected_total': 5, 'issues_truncated': True, 'action': 'truncate_report'})]

OK - Grupo 2.7.2 - apply_issue_truncation con truncamiento


### Grupo 2.8 - build_validation_summary

Qué se prueba:
Que el summary mínimo y estable quede bien armado:

- conteos por nivel,
- conteos por code,
- checks ejecutados,
- campos evaluados,
- y bloques opcionales cuando se pasan.


#### 2.8.1 - summary mínimo sin opcionales

In [49]:
schema_summary = make_trip_schema([
    make_field("movement_id", "string", required=True),
    make_field("mode", "categorical", required=False),
])

issues_summary_in = [
    Issue(level="warning", code="VAL.DOMAIN.PARTIAL_COVERAGE", message="warn 1"),
    Issue(level="warning", code="VAL.DOMAIN.PARTIAL_COVERAGE", message="warn 2"),
    Issue(level="error", code="VAL.CORE.NULLABILITY_VIOLATION", message="err 1"),
]

summary = build_validation_summary(
    n_rows=10,
    issues=issues_summary_in,
    schema=schema_summary,
    checks_executed={
        "required_fields": True,
        "types_and_formats": True,
        "constraints": True,
        "domains": True,
        "temporal_consistency": False,
        "duplicates": False,
    },
    checked_fields=["movement_id", "mode"],
)

assert summary["ok"] is False
assert summary["n_rows"] == 10
assert summary["n_issues"] == 3
assert summary["n_errors"] == 1
assert summary["n_warnings"] == 2
assert summary["n_info"] == 0
assert summary["counts_by_level"] == {"error": 1, "warning": 2, "info": 0}
assert summary["counts_by_code"]["VAL.DOMAIN.PARTIAL_COVERAGE"] == 2
assert summary["counts_by_code"]["VAL.CORE.NULLABILITY_VIOLATION"] == 1
assert summary["checked_fields"] == ["movement_id", "mode"]
assert summary["schema_version"] == "1.1"

assert_json_safe(summary, "validation_summary_minimal")

display(summary)
show_ok("Grupo 2.8.1 - build_validation_summary mínimo")

{'ok': False,
 'n_rows': 10,
 'n_issues': 3,
 'n_errors': 1,
 'n_warnings': 2,
 'n_info': 0,
 'counts_by_level': {'error': 1, 'warning': 2, 'info': 0},
 'counts_by_code': {'VAL.DOMAIN.PARTIAL_COVERAGE': 2,
  'VAL.CORE.NULLABILITY_VIOLATION': 1},
 'checked_fields': ['movement_id', 'mode'],
 'checks_executed': {'required_fields': True,
  'types_and_formats': True,
  'constraints': True,
  'domains': True,
  'temporal_consistency': False,
  'duplicates': False},
 'schema_version': '1.1'}

OK - Grupo 2.8.1 - build_validation_summary mínimo


#### 2.8.2 - summary con opcionales

In [50]:
summary = build_validation_summary(
    n_rows=5,
    issues=[],
    schema=schema_summary,
    checks_executed={
        "required_fields": True,
        "types_and_formats": True,
        "constraints": True,
        "domains": True,
        "temporal_consistency": True,
        "duplicates": True,
    },
    checked_fields=["movement_id", "mode"],
    domains_block={
        "mode": "full",
        "min_required_ratio": 1.0,
        "fields": {"mode": {"ratio_in_domain": 1.0}},
    },
    temporal_block={
        "evaluated": True,
        "tier": "tier_1",
        "n_checked": 5,
        "n_violations": 0,
    },
    duplicates_block={
        "evaluated": True,
        "duplicates_subset": ["user_id", "origin_time_utc"],
        "n_duplicate_rows": 0,
    },
    limits_block={
        "max_issues": 500,
        "issues_truncated": False,
        "n_issues_emitted": 0,
        "n_issues_detected_total": 0,
    },
)

assert summary["domains"]["mode"] == "full"
assert summary["temporal"]["evaluated"] is True
assert summary["duplicates"]["n_duplicate_rows"] == 0
assert summary["limits"]["issues_truncated"] is False

assert_json_safe(summary, "validation_summary_with_optionals")

show_ok("Grupo 2.8.2 - build_validation_summary con opcionales")

OK - Grupo 2.8.2 - build_validation_summary con opcionales


In [51]:
show_ok("Sección 2 completa - Helpers de chequeo propiamente tales")

OK - Sección 2 completa - Helpers de chequeo propiamente tales


## Bloque 3: Smoke tests de validate_trips

### Grupo 3.1 - Fixtures minimos reutilizables

In [52]:
from copy import deepcopy
from types import SimpleNamespace

import pandas as pd

from pylondrina.schema import FieldSpec, DomainSpec, TripSchema
from pylondrina.datasets import TripDataset
from pylondrina.errors import ValidationError, SchemaError
from pylondrina.validation import ValidationOptions, validate_trips


def make_field(
    name: str,
    dtype: str,
    *,
    required: bool = False,
    constraints: dict | None = None,
    domain: DomainSpec | None = None,
) -> FieldSpec:
    return FieldSpec(
        name=name,
        dtype=dtype,
        required=required,
        constraints=constraints,
        domain=domain,
    )


BASE_VALIDATE_SCHEMA = TripSchema(
    version="1.1",
    fields={
        "movement_id": make_field("movement_id", "string", required=True),
        "user_id": make_field("user_id", "string", required=True),

        # Coordenadas OD: las dejamos como required para que el smoke test
        # realmente atraviese la regla especial allow_partial_od_spatial.
        "origin_latitude": make_field("origin_latitude", "float", required=True),
        "origin_longitude": make_field("origin_longitude", "float", required=True),
        "destination_latitude": make_field("destination_latitude", "float", required=True),
        "destination_longitude": make_field("destination_longitude", "float", required=True),

        "origin_time_utc": make_field("origin_time_utc", "datetime", required=False),
        "destination_time_utc": make_field("destination_time_utc", "datetime", required=False),

        "mode": make_field(
            "mode",
            "categorical",
            required=False,
            domain=DomainSpec(values=["walk", "bus", "metro", "unknown"], extendable=False),
        ),
        "purpose": make_field(
            "purpose",
            "categorical",
            required=False,
            domain=DomainSpec(values=["work", "study", "health", "unknown"], extendable=False),
        ),
    },
    required=[
        "movement_id",
        "user_id",
        "origin_latitude",
        "origin_longitude",
        "destination_latitude",
        "destination_longitude",
    ],
    semantic_rules=None,
)


BASE_SCHEMA_EFFECTIVE = SimpleNamespace(
    domains_effective={
        "mode": {
            "values": ["walk", "bus", "metro", "unknown"],
            "extendable": False,
            "extended": False,
            "added_values": [],
            "unknown_value": "unknown",
        },
        "purpose": {
            "values": ["work", "study", "health", "unknown"],
            "extendable": False,
            "extended": False,
            "added_values": [],
            "unknown_value": "unknown",
        },
    },
    temporal={"tier": "tier_1"},
)


def make_tripdataset_for_validate(
    df: pd.DataFrame,
    *,
    schema: TripSchema = BASE_VALIDATE_SCHEMA,
    schema_effective=BASE_SCHEMA_EFFECTIVE,
    metadata: dict | None = None,
) -> TripDataset:
    base_metadata = {
        "dataset_id": "ds-smoke-validate-001",
        "events": [],
        "is_validated": False,
        "domains_effective": deepcopy(getattr(schema_effective, "domains_effective", {})),
        "temporal": {"tier": "tier_1", "fields_present": ["origin_time_utc", "destination_time_utc"]},
    }
    if metadata:
        base_metadata.update(deepcopy(metadata))

    return TripDataset(
        data=df.copy(),
        schema=schema,
        schema_version=schema.version,
        provenance={
            "source": {
                "name": "synthetic",
                "entity": "trips",
                "version": "helper-smoke-validate-v1",
            },
            "notes": ["dataset sintético para smoke tests integrados de validate_trips"],
        },
        field_correspondence={},
        value_correspondence={},
        metadata=base_metadata,
        schema_effective=schema_effective,
    )


VALIDATE_OPTIONS_DEFAULT = ValidationOptions()

VALIDATE_OPTIONS_STRICT = ValidationOptions(
    strict=True,
    validate_temporal_consistency=True,
)

VALIDATE_OPTIONS_DOMAINS_FULL = ValidationOptions(
    validate_domains="full",
)

VALIDATE_OPTIONS_DUPLICATES = ValidationOptions(
    validate_duplicates=True,
    duplicates_subset=("user_id", "origin_time_utc", "origin_longitude", "destination_longitude"),
)

### Grupo 3.2 - happy path mínimo integrado

Qué prueba: la integración básica de la función pública en caso correcto: report, evento, issues_summary, is_validated y no mutación de trips.data. El contrato vigente exige exactamente esos side effects y preservación del dataframe.

In [57]:
df_ok = pd.DataFrame({
    "movement_id": ["m1", "m2"],
    "user_id": ["u1", "u2"],
    "origin_latitude": [-33.45, -33.46],
    "origin_longitude": [-70.66, -70.67],
    "destination_latitude": [-33.43, -33.44],
    "destination_longitude": [-70.61, -70.62],
    "origin_time_utc": ["2026-04-01T08:00:00Z", "2026-04-01T09:00:00Z"],
    "destination_time_utc": ["2026-04-01T08:30:00Z", "2026-04-01T09:20:00Z"],
    "mode": ["walk", "bus"],
    "purpose": ["work", "study"],
})

trips = make_tripdataset_for_validate(df_ok)
data_before = trips.data.copy(deep=True)

report = validate_trips(trips, options=VALIDATE_OPTIONS_DEFAULT)

assert isinstance(report, ValidationReport)
assert report.ok is True
assert len(report.issues) == 0
assert trips.metadata["is_validated"] is True
assert len(trips.metadata["events"]) == 1

event = trips.metadata["events"][-1]
assert event["op"] == "validate_trips"
assert event["summary"] == report.summary
assert "parameters" in event
assert "issues_summary" in event

pd.testing.assert_frame_equal(trips.data, data_before)

display(report)
show_ok("Grupo 3.2 - happy path mínimo integrado")

ValidationReport(ok=True, issues=[], summary={'ok': True, 'n_rows': 2, 'n_issues': 0, 'n_errors': 0, 'n_warnings': 0, 'n_info': 0, 'counts_by_level': {'error': 0, 'warning': 0, 'info': 0}, 'counts_by_code': {}, 'checked_fields': ['destination_latitude', 'destination_longitude', 'destination_time_utc', 'mode', 'movement_id', 'origin_latitude', 'origin_longitude', 'origin_time_utc', 'purpose', 'user_id'], 'checks_executed': {'required_fields': True, 'types_and_formats': True, 'constraints': True, 'domains': False, 'temporal_consistency': False, 'duplicates': False}, 'schema_version': '1.1', 'limits': {'max_issues': 500, 'issues_truncated': False, 'n_issues_emitted': 0, 'n_issues_detected_total': 0}}, parameters=None)

OK - Grupo 3.2 - happy path mínimo integrado


### Grupo 3.3 - regla especial de OD parcial espacial

Qué prueba: la excepción de allow_partial_od_spatial=True: basta con origen completo o destino completo; solo falla si faltan ambos extremos. Esa regla quedó cerrada como parte normativa del contrato de OP-02.

In [58]:
df_partial_od = pd.DataFrame({
    "movement_id": ["m1", "m2", "m3"],
    "user_id": ["u1", "u2", "u3"],

    # Fila 1: origen completo, destino ausente -> válida por excepción
    "origin_latitude": [-33.45, None, None],
    "origin_longitude": [-70.66, None, None],

    # Fila 2: destino completo, origen ausente -> válida por excepción
    "destination_latitude": [None, -33.44, None],
    "destination_longitude": [None, -70.62, None],

    "origin_time_utc": ["2026-04-01T08:00:00Z"] * 3,
    "destination_time_utc": ["2026-04-01T08:30:00Z"] * 3,
    "mode": ["walk", "bus", "metro"],
    "purpose": ["work", "study", "health"],
})

trips = make_tripdataset_for_validate(df_partial_od)

report = validate_trips(
    trips,
    options=ValidationOptions(
        validate_constraints=True,
        allow_partial_od_spatial=True,
    ),
)

codes = [i.code for i in report.issues]

assert report.ok is False
assert "VAL.CORE.OD_SPATIAL_BOTH_MISSING" in codes
assert trips.metadata["is_validated"] is False
assert len(trips.metadata["events"]) == 1

display(report.issues)
show_ok("Grupo 3.3 - OD parcial espacial")

[Issue(level='error', code='VAL.CORE.OD_SPATIAL_BOTH_MISSING', message='La regla de OD parcial falló: la fila no tiene ni origen espacial completo ni destino espacial completo en coordenadas.', field=None, source_field=None, row_count=1, details={'check': 'constraints', 'n_rows_total': 3, 'n_violations': 1, 'row_indices_sample': [2], 'values_sample': [{'origin_latitude': None, 'origin_longitude': None, 'destination_latitude': None, 'destination_longitude': None}], 'raw_values_sample': [{'origin_latitude': None, 'origin_longitude': None, 'destination_latitude': None, 'destination_longitude': None}], 'fields_checked': ['origin_latitude', 'origin_longitude', 'destination_latitude', 'destination_longitude'], 'allow_partial_od_spatial': True, 'action': 'report_error'})]

OK - Grupo 3.3 - OD parcial espacial


### Grupo 3.4 - warning no fatal por constraint conocido mal formado

Qué prueba: si un constraint es conocido pero sus params vienen mal formados, corresponde warning + skip, no fatal. Esa alerta quedó cerrada explícitamente en el contrato aterrizado.

In [60]:
schema_warn = TripSchema(
    version="1.1",
    fields={
        **BASE_VALIDATE_SCHEMA.fields,
        "score": make_field(
            "score",
            "float",
            required=False,
            constraints={"range": {"minimum": 0}},  # mal formado: debería usar min/max
        ),
    },
    required=list(BASE_VALIDATE_SCHEMA.required),
    semantic_rules=None,
)

df_warn = pd.DataFrame({
    "movement_id": ["m1"],
    "user_id": ["u1"],
    "origin_latitude": [-33.45],
    "origin_longitude": [-70.66],
    "destination_latitude": [-33.43],
    "destination_longitude": [-70.61],
    "origin_time_utc": ["2026-04-01T08:00:00Z"],
    "destination_time_utc": ["2026-04-01T08:30:00Z"],
    "mode": ["walk"],
    "purpose": ["work"],
    "score": [12.5],
})

trips = make_tripdataset_for_validate(df_warn, schema=schema_warn)

report = validate_trips(
    trips,
    options=ValidationOptions(validate_constraints=True),
)

codes = [i.code for i in report.issues]
levels = [i.level for i in report.issues]

assert "VAL.SCHEMA.CONSTRAINT_PARAMS_INVALID" in codes
assert "warning" in levels
assert trips.metadata["is_validated"] is True
assert len(trips.metadata["events"]) == 1

display(report.issues)
show_ok("Grupo 3.4 - warning no fatal por constraint mal formado")

[Issue(level='warning', code='VAL.SCHEMA.CONSTRAINT_PARAMS_INVALID', message="La constraint 'range' del campo 'score' tiene parámetros inválidos o incompletos; se omitirá solo esa constraint.", field='score', source_field=None, row_count=None, details={'check': 'constraints', 'field': 'score', 'constraint': 'range', 'expected_params': ['min', 'max'], 'received_params': ['minimum'], 'reason': 'invalid_or_incomplete_params', 'action': 'skip_constraint'})]

OK - Grupo 3.4 - warning no fatal por constraint mal formado


### Grupo 3.5 - strict=True con error de datos

Qué prueba: si hay error de datos y strict=True, primero se deja evidencia (report implícito en evento/summary y is_validated=False) y recién después se lanza. Esa es una política central de OP-02.

In [63]:
df_strict = pd.DataFrame({
    "movement_id": ["m1"],
    "user_id": ["u1"],
    "origin_latitude": [-33.45],
    "origin_longitude": [-70.66],
    "destination_latitude": [-33.43],
    "destination_longitude": [-70.61],
    "origin_time_utc": ["2026-04-01T09:00:00Z"],
    "destination_time_utc": ["2026-04-01T08:30:00Z"],  # inconsistencia temporal
    "mode": ["walk"],
    "purpose": ["work"],
})

trips = make_tripdataset_for_validate(df_strict)

raised = None
try:
    validate_trips(
        trips,
        options=ValidationOptions(
            strict=True,
            validate_temporal_consistency=True,
        ),
    )
except Exception as e:
    raised = e

assert raised is not None
assert isinstance(raised, ValidationError)

assert trips.metadata["is_validated"] is False
assert len(trips.metadata["events"]) == 1

event = trips.metadata["events"][-1]
assert event["op"] == "validate_trips"
assert "issues_summary" in event

display(raised)
show_ok("Grupo 3.5 - strict=True con error de datos")

ValidationError(message='validate_trips detectó errores de datos y strict=True exige abortar.', code='VAL.TEMPORAL.ORIGIN_AFTER_DESTINATION', details={'check': 'temporal_consistency', 'n_rows_total': 1, 'n_violations': 1, 'row_indices_sample': [0], 'values_sample': [{'origin_time_utc': '2026-04-01T09:00:00Z', 'destination_time_utc': '2026-04-01T08:30:00Z'}], 'raw_values_sample': [{'origin_time_utc': '2026-04-01T09:00:00Z', 'destination_time_utc': '2026-04-01T08:30:00Z'}], 'origin_field': 'origin_time_utc', 'destination_field': 'destination_time_utc', 'action': 'report_error'}, issue=Issue(level='error', code='VAL.TEMPORAL.ORIGIN_AFTER_DESTINATION', message='Se detectaron filas donde origin_time_utc es posterior a destination_time_utc.', field=None, source_field=None, row_count=1, details={'check': 'temporal_consistency', 'n_rows_total': 1, 'n_violations': 1, 'row_indices_sample': [0], 'values_sample': [{'origin_time_utc': '2026-04-01T09:00:00Z', 'destination_time_utc': '2026-04-01T08:3

OK - Grupo 3.5 - strict=True con error de datos


### Grupo 3.6 - fatal de config sin evento

Qué prueba: abort fatal de config/schema antes de correr el pipeline normal. En este caso, validate_duplicates=True sin duplicates_subset usable. Aquí no debe haber report ni evento.

In [66]:
df_fatal = pd.DataFrame({
    "movement_id": ["m1"],
    "user_id": ["u1"],
    "origin_latitude": [-33.45],
    "origin_longitude": [-70.66],
    "destination_latitude": [-33.43],
    "destination_longitude": [-70.61],
    "origin_time_utc": ["2026-04-01T08:00:00Z"],
    "destination_time_utc": ["2026-04-01T08:30:00Z"],
    "mode": ["walk"],
    "purpose": ["work"],
})

trips = make_tripdataset_for_validate(df_fatal)
events_before = deepcopy(trips.metadata["events"])
validated_before = trips.metadata["is_validated"]

raised = None
try:
    validate_trips(
        trips,
        options=ValidationOptions(
            validate_duplicates=True,
            duplicates_subset=None,
        ),
    )
except Exception as e:
    raised = e

assert raised is not None
assert isinstance(raised, ValidationError)

assert trips.metadata["events"] == events_before
assert trips.metadata["is_validated"] == validated_before

display(raised)
show_ok("Grupo 3.6 - fatal de config sin evento")

ValidationError(message='validate_duplicates=True requiere duplicates_subset explícito; no se definió subset usable.', code='VAL.CONFIG.DUPLICATES_SUBSET_NOT_PROVIDED', details={'validate_duplicates': True, 'duplicates_subset': None, 'action': 'abort'}, issue=Issue(level='error', code='VAL.CONFIG.DUPLICATES_SUBSET_NOT_PROVIDED', message='validate_duplicates=True requiere duplicates_subset explícito; no se definió subset usable.', field=None, source_field=None, row_count=None, details={'validate_duplicates': True, 'duplicates_subset': None, 'action': 'abort'}), issues=(Issue(level='error', code='VAL.CONFIG.DUPLICATES_SUBSET_NOT_PROVIDED', message='validate_duplicates=True requiere duplicates_subset explícito; no se definió subset usable.', field=None, source_field=None, row_count=None, details={'validate_duplicates': True, 'duplicates_subset': None, 'action': 'abort'}),))

OK - Grupo 3.6 - fatal de config sin evento


### Grupo 3.7 - truncamiento + consistencia de summary/evento

Qué prueba: truncamiento explícito con issue final y bloque limits en summary, además de consistencia entre report y evento. 

In [67]:
df_trunc = pd.DataFrame({
    "movement_id": [None, "m2", "m3", "m4"],
    "user_id": ["u1", "u2", "u3", "u4"],
    "origin_latitude": [-33.45, -33.45, -33.45, -33.45],
    "origin_longitude": [-70.66, -70.66, -70.66, -70.66],
    "destination_latitude": [-33.43, -33.43, -33.43, -33.43],
    "destination_longitude": [-70.61, -70.61, -70.61, -70.61],
    "origin_time_utc": [
        "2026-04-01T09:00:00Z",
        "2026-04-01T09:00:00Z",
        "2026-04-01T09:00:00Z",
        "2026-04-01T09:00:00Z",
    ],
    "destination_time_utc": [
        "2026-04-01T08:00:00Z",  # temporal error
        "2026-04-01T08:00:00Z",  # temporal error
        "2026-04-01T08:00:00Z",  # temporal error
        "2026-04-01T08:00:00Z",  # temporal error
    ],
    "mode": ["xxx", "yyy", "walk", "bus"],       # dominio fuera
    "purpose": ["zzz", "work", "study", "aaa"],  # dominio fuera
})

trips = make_tripdataset_for_validate(df_trunc)

report = validate_trips(
    trips,
    options=ValidationOptions(
        max_issues=2,
        validate_temporal_consistency=True,
        validate_domains="full",
    ),
)

codes = [i.code for i in report.issues]

assert "VAL.CORE.ISSUES_TRUNCATED" in codes
assert report.summary["limits"]["issues_truncated"] is True
assert report.summary["limits"]["n_issues_emitted"] <= 2
assert report.summary["limits"]["n_issues_detected_total"] >= report.summary["limits"]["n_issues_emitted"]

event = trips.metadata["events"][-1]
assert event["summary"] == report.summary
assert "issues_summary" in event

show_ok("Grupo 3.7 - truncamiento + consistency de summary/evento")

OK - Grupo 3.7 - truncamiento + consistency de summary/evento
